# Cloud Core - BTC Scalping Signal Engine

**4-Layer Ensemble untuk Signal Generation**

- Layer 1: BCD (Bayesian Changepoint Detection) - Regime detection
- Layer 2: EMA - Trend confirmation
- Layer 3: AI (MLP/XGBoost) - Next-candle predictor (Gatekeeper)
- Layer 4: Risk - ATR-based position sizing

Run di Google Colab atau Jupyter locally.

## 1. Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn xgboost hmmlearn pandas-ta ccxt python-dotenv -q

## 2. Layer 1: BCD (Bayesian Changepoint Detection)

In [ ]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from typing import Tuple, Optional

class BayesianChangepointModel:
    N_STATES = 4
    RANDOM_STATE = 42
    
    def __init__(self):
        self.model = None
        self._is_fitted = False
    
    def fit(self, df: pd.DataFrame) -> bool:
        try:
            df = df.copy()
            df['returns'] = np.log(df['Close'] / df['Close'].shift(1))
            df['volatility'] = df['returns'].rolling(20).std()
            df['price_range'] = (df['High'] - df['Low']) / df['Close']
            features = df[['returns', 'volatility', 'price_range']].dropna()
            
            if len(features) < 100:
                return False
            
            self.model = GaussianHMM(
                n_components=self.N_STATES,
                covariance_type="full",
                random_state=self.RANDOM_STATE,
                n_iter=100
            )
            self.model.fit(features.values)
            self._is_fitted = True
            return True
        except Exception as e:
            print(f"[BCD] Fit error: {e}")
            return False
    
    def get_directional_vote(self, df: pd.DataFrame) -> float:
        try:
            if not self._is_fitted or self.model is None:
                # Fallback
                ema20 = df['Close'].ewm(span=20).mean().iloc[-1]
                ema50 = df['Close'].ewm(span=50).mean().iloc[-1]
                price = df['Close'].iloc[-1]
                if price > ema20 > ema50:
                    return 0.6
                elif price < ema20 < ema50:
                    return -0.6
                return 0.0
            
            df = df.copy()
            df['returns'] = np.log(df['Close'] / df['Close'].shift(1))
            df['volatility'] = df['returns'].rolling(20).std()
            df['price_range'] = (df['High'] - df['Low']) / df['Close']
            features = df[['returns', 'volatility', 'price_range']].dropna()
            
            if len(features) == 0:
                return 0.0
            
            states = self.model.predict(features.values)
            current_state = states[-1]
            
            means = self.model.means_
            state_returns = [(i, means[i][0]) for i in range(self.N_STATES)]
            state_returns.sort(key=lambda x: x[1], reverse=True)
            
            bull_states = [s[0] for s in state_returns[:2]]
            bear_states = [s[0] for s in state_returns[2:]]
            
            if current_state in bull_states:
                return 0.7
            elif current_state in bear_states:
                return -0.7
            return 0.0
            
        except Exception as e:
            print(f"[BCD] Vote error: {e}")
            return 0.0

## 3. Layer 2: EMA Signal Model

In [ ]:
class EMASignalModel:
    def get_directional_vote(self, df: pd.DataFrame) -> float:
        if len(df) < 200:
            return 0.0
        
        ema20 = df['Close'].ewm(span=20, adjust=False).mean()
        ema50 = df['Close'].ewm(span=50, adjust=False).mean()
        ema200 = df['Close'].ewm(span=200, adjust=False).mean()
        
        close = df['Close'].iloc[-1]
        e20 = ema20.iloc[-1]
        e50 = ema50.iloc[-1]
        e200 = ema200.iloc[-1]
        
        bullish = close > e20 > e50 > e200
        bearish = close < e20 < e50 < e200
        
        e20_prev = ema20.iloc[-5] if len(ema20) >= 5 else e20
        ema_rising = e20 > e20_prev
        
        if bullish and ema_rising:
            return 1.0
        elif bullish:
            return 0.6
        elif bearish and not ema_rising:
            return -1.0
        elif bearish:
            return -0.6
        elif close > e200:
            return 0.3
        elif close < e200:
            return -0.3
        return 0.0

## 4. Layer 3: MLP Neural Network

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

class MLPSignalModel:
    MIN_ROWS = 60
    HIDDEN_LAYERS = (128, 64)
    FEATURE_COLS = ["rsi_14", "macd_hist", "ema20_dist", "log_return", "norm_atr", "funding", "oi_change"]
    
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self._is_trained = False
    
    def _prepare_features(self, df):
        import pandas_ta as ta
        df = df.copy()
        df['rsi_14'] = ta.rsi(df['Close'], length=14)
        
        macd_df = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        if macd_df is not None:
            df['macd_hist'] = macd_df['MACDh_12_26_9']
        else:
            df['macd_hist'] = 0.0
        
        ema20 = ta.ema(df['Close'], length=20)
        df['ema20_dist'] = (df['Close'] - ema20) / ema20
        df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
        
        atr = ta.atr(df['High'], df['Low'], df['Close'], length=14)
        df['norm_atr'] = atr / df['Close']
        
        df['funding'] = df.get('Funding', 0.0)
        df['oi_change'] = 0.0
        
        return df
    
    def train(self, df):
        try:
            if len(df) < self.MIN_ROWS:
                return False
            
            df_feat = self._prepare_features(df)
            
            # Target: forward return
            df_feat['future_close'] = df_feat['Close'].shift(-1)
            df_feat['price_move_pct'] = (df_feat['future_close'] - df_feat['Close']) / df_feat['Close']
            df_feat['threshold'] = 0.5 * df_feat['norm_atr']
            
            move = df_feat['price_move_pct'].values
            thr = df_feat['threshold'].values
            labels = np.where(move > thr, 2, np.where(move < -thr, 0, 1))
            labels = np.where(np.isnan(move) | np.isnan(thr), np.nan, labels)
            df_feat['target'] = labels
            
            df_train = df_feat.dropna(subset=self.FEATURE_COLS + ['target'])
            
            if len(df_train) < 50:
                return False
            
            X = df_train[self.FEATURE_COLS].values
            y = df_train['target'].values.astype(int)
            
            X_scaled = self.scaler.fit_transform(X)
            
            self.model = MLPClassifier(
                hidden_layer_sizes=self.HIDDEN_LAYERS,
                activation='relu',
                solver='adam',
                max_iter=300,
                early_stopping=True,
                validation_fraction=0.15,
                random_state=42
            )
            self.model.fit(X_scaled, y)
            self._is_trained = True
            return True
            
        except Exception as e:
            print(f"[MLP] Train error: {e}")
            return False
    
    def predict(self, df):
        if not self._is_trained or self.model is None:
            return 'NEUTRAL', 50.0, np.array([0.33, 0.34, 0.33])
        
        try:
            df_feat = self._prepare_features(df)
            latest = df_feat[self.FEATURE_COLS].iloc[[-1]]
            
            if latest.isna().any().any():
                return 'NEUTRAL', 50.0, np.array([0.33, 0.34, 0.33])
            
            X_latest = self.scaler.transform(latest.values)
            probs = self.model.predict_proba(X_latest)[0]
            
            class_map = {c: i for i, c in enumerate(self.model.classes_)}
            prob_bear = probs[class_map.get(0, 0)] if 0 in class_map else 0.0
            prob_neut = probs[class_map.get(1, 1)] if 1 in class_map else 0.0
            prob_bull = probs[class_map.get(2, 2)] if 2 in class_map else 0.0
            
            if prob_bull > prob_bear and prob_bull > prob_neut:
                return 'BULL', round(prob_bull * 100, 1), np.array([prob_bear, prob_neut, prob_bull])
            elif prob_bear > prob_bull and prob_bear > prob_neut:
                return 'BEAR', round(prob_bear * 100, 1), np.array([prob_bear, prob_neut, prob_bull])
            else:
                return 'NEUTRAL', round(max(prob_neut * 100, 50.0), 1), np.array([prob_bear, prob_neut, prob_bull])
                
        except Exception as e:
            print(f"[MLP] Predict error: {e}")
            return 'NEUTRAL', 50.0, np.array([0.33, 0.34, 0.33])
    
    def get_directional_vote(self, df):
        bias, conf, probs = self.predict(df)
        if bias == 'BULL':
            return (conf - 50) / 50
        elif bias == 'BEAR':
            return -(conf - 50) / 50
        return 0.0

## 5. Layer 3 Alternative: XGBoost

In [ ]:
import xgboost as xgb

class XGBoostSignalModel:
    MIN_ROWS = 60
    FEATURE_COLS = ["rsi_14", "macd_hist", "ema20_dist", "log_return", "norm_atr", "funding", "oi_change"]
    
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self._is_trained = False
    
    def _prepare_features(self, df):
        # Same as MLP
        import pandas_ta as ta
        df = df.copy()
        df['rsi_14'] = ta.rsi(df['Close'], length=14)
        
        macd_df = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        df['macd_hist'] = macd_df['MACDh_12_26_9'] if macd_df is not None else 0.0
        
        ema20 = ta.ema(df['Close'], length=20)
        df['ema20_dist'] = (df['Close'] - ema20) / ema20
        df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
        
        atr = ta.atr(df['High'], df['Low'], df['Close'], length=14)
        df['norm_atr'] = atr / df['Close']
        
        df['funding'] = df.get('Funding', 0.0)
        df['oi_change'] = 0.0
        
        return df
    
    def train(self, df):
        try:
            if len(df) < self.MIN_ROWS:
                return False
            
            df_feat = self._prepare_features(df)
            
            df_feat['future_close'] = df_feat['Close'].shift(-1)
            df_feat['price_move_pct'] = (df_feat['future_close'] - df_feat['Close']) / df_feat['Close']
            df_feat['threshold'] = 0.5 * df_feat['norm_atr']
            
            move = df_feat['price_move_pct'].values
            thr = df_feat['threshold'].values
            labels = np.where(move > thr, 2, np.where(move < -thr, 0, 1))
            labels = np.where(np.isnan(move) | np.isnan(thr), np.nan, labels)
            df_feat['target'] = labels
            
            df_train = df_feat.dropna(subset=self.FEATURE_COLS + ['target'])
            
            if len(df_train) < 50:
                return False
            
            X = df_train[self.FEATURE_COLS].values
            y = df_train['target'].values.astype(int)
            
            X_scaled = self.scaler.fit_transform(X)
            
            self.model = xgb.XGBClassifier(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                objective='multi:softprob',
                num_class=3,
                eval_metric='mlogloss',
                random_state=42,
                use_label_encoder=False
            )
            self.model.fit(X_scaled, y)
            self._is_trained = True
            return True
            
        except Exception as e:
            print(f"[XGB] Train error: {e}")
            return False
    
    def predict(self, df):
        if not self._is_trained or self.model is None:
            return 'NEUTRAL', 50.0, np.array([0.33, 0.34, 0.33])
        
        try:
            df_feat = self._prepare_features(df)
            latest = df_feat[self.FEATURE_COLS].iloc[[-1]]
            
            if latest.isna().any().any():
                return 'NEUTRAL', 50.0, np.array([0.33, 0.34, 0.33])
            
            X_latest = self.scaler.transform(latest.values)
            probs = self.model.predict_proba(X_latest)[0]
            
            prob_bear, prob_neut, prob_bull = probs[0], probs[1], probs[2]
            
            if prob_bull > prob_bear and prob_bull > prob_neut:
                return 'BULL', round(prob_bull * 100, 1), probs
            elif prob_bear > prob_bull and prob_bear > prob_neut:
                return 'BEAR', round(prob_bear * 100, 1), probs
            else:
                return 'NEUTRAL', round(max(prob_neut * 100, 50.0), 1), probs
                
        except Exception as e:
            print(f"[XGB] Predict error: {e}")
            return 'NEUTRAL', 50.0, np.array([0.33, 0.34, 0.33])
    
    def get_directional_vote(self, df):
        bias, conf, _ = self.predict(df)
        if bias == 'BULL':
            return (conf - 50) / 50
        elif bias == 'BEAR':
            return -(conf - 50) / 50
        return 0.0

## 6. Layer 4: Risk Model (ATR-based)

In [ ]:
class RiskModel:
    def get_multiplier(self, df):
        try:
            import pandas_ta as ta
            if len(df) < 14:
                return 0.5
            
            atr = ta.atr(df['High'], df['Low'], df['Close'], length=14)
            close = df['Close'].iloc[-1]
            vol_ratio = atr.iloc[-1] / close if not pd.isna(atr.iloc[-1]) else 0.02
            
            if vol_ratio < 0.008:
                return 1.0
            elif vol_ratio < 0.012:
                return 0.8
            elif vol_ratio < 0.015:
                return 0.5
            elif vol_ratio < 0.020:
                return 0.2
            return 0.0
        except:
            return 0.5
    
    def get_risk_params(self, df, base_sl_pct=1.5):
        try:
            import pandas_ta as ta
            atr = ta.atr(df['High'], df['Low'], df['Close'], length=14)
            vol_ratio = atr.iloc[-1] / df['Close'].iloc[-1]
        except:
            vol_ratio = 0.012
        
        l4_mult = self.get_multiplier(df)
        
        if vol_ratio < 0.008:
            sl_pct = base_sl_pct * 0.8
        elif vol_ratio < 0.012:
            sl_pct = base_sl_pct
        elif vol_ratio < 0.015:
            sl_pct = base_sl_pct * 1.2
        elif vol_ratio < 0.020:
            sl_pct = base_sl_pct * 1.5
        else:
            sl_pct = base_sl_pct * 2.0
        
        tp_pct = sl_pct * 2
        
        if vol_ratio < 0.010:
            leverage = 20
        elif vol_ratio < 0.015:
            leverage = 15
        elif vol_ratio < 0.020:
            leverage = 10
        else:
            leverage = 5
        
        return {
            'l4_multiplier': l4_mult,
            'sl_pct': round(sl_pct, 2),
            'tp_pct': round(tp_pct, 2),
            'leverage': leverage
        }

## 7. Directional Spectrum (Aggregator)

In [ ]:
from dataclasses import dataclass

@dataclass
class SpectrumResult:
    directional_bias: float
    conviction_pct: float
    action: str
    trade_gate: str
    position_size_pct: float
    weights: dict

class DirectionalSpectrum:
    WEIGHTS = {
        'l1_bcd': 0.30,
        'l2_ema': 0.25,
        'l3_ai': 0.45,
    }
    
    GATE_THRESHOLD = 0.20
    ADVISORY_MIN = 0.15
    BASE_SIZE = 5.0
    MAX_SIZE = 25.0
    
    def calculate(self, l1_vote, l2_vote, l3_vote, l4_mult, base_size=5.0):
        raw_score = (
            l1_vote * self.WEIGHTS['l1_bcd'] +
            l2_vote * self.WEIGHTS['l2_ema'] +
            l3_vote * self.WEIGHTS['l3_ai']
        )
        
        directional_bias = raw_score * l4_mult
        
        abs_score = abs(directional_bias)
        conviction_pct = min(abs_score * 200, 100.0)
        
        if directional_bias > 0.05:
            action = 'LONG'
        elif directional_bias < -0.05:
            action = 'SHORT'
        else:
            action = 'NEUTRAL'
        
        if abs_score >= self.GATE_THRESHOLD:
            trade_gate = 'ACTIVE'
        elif abs_score >= self.ADVISORY_MIN:
            trade_gate = 'ADVISORY'
        else:
            trade_gate = 'SUSPENDED'
        
        if trade_gate == 'ACTIVE':
            position_size_pct = base_size * abs_score / self.GATE_THRESHOLD
        elif trade_gate == 'ADVISORY':
            position_size_pct = base_size * 0.5
        else:
            position_size_pct = 0.0
        
        position_size_pct = min(position_size_pct, self.MAX_SIZE)
        
        return SpectrumResult(
            directional_bias=round(directional_bias, 4),
            conviction_pct=round(conviction_pct, 2),
            action=action,
            trade_gate=trade_gate,
            position_size_pct=round(position_size_pct, 2),
            weights=self.WEIGHTS
        )

## 8. Data Fetcher (Binance)

In [ ]:
import ccxt

class DataFetcher:
    def __init__(self, exchange_id='binance'):
        self.exchange = getattr(ccxt, exchange_id)({
            'enableRateLimit': True,
            'options': {'defaultType': 'future'}
        })
    
    def fetch_ohlcv(self, symbol='BTC/USDT', timeframe='4h', limit=500):
        try:
            print(f"[DataFetcher] Fetching {limit} {timeframe} candles for {symbol}")
            ohlcv = self.exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
            
            df = pd.DataFrame(
                ohlcv,
                columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']
            )
            df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='ms')
            df.set_index('Timestamp', inplace=True)
            
            print(f"[DataFetcher] Fetched {len(df)} candles")
            return df
            
        except Exception as e:
            print(f"[DataFetcher] Error: {e}")
            return None

## 9. Signal Service (Orchestrator)

In [ ]:
from datetime import datetime

@dataclass
class SignalResult:
    timestamp: datetime
    symbol: str
    price: float
    l1_vote: float
    l2_vote: float
    l3_vote: float
    l4_mult: float
    directional_bias: float
    action: str
    conviction_pct: float
    trade_gate: str
    position_size_pct: float
    sl_pct: float
    tp_pct: float
    leverage: int
    model_used: str

class SignalService:
    def __init__(self, ai_model='mlp'):
        self.ai_model_type = ai_model
        self.l1_bcd = BayesianChangepointModel()
        self.l2_ema = EMASignalModel()
        self.l4_risk = RiskModel()
        self.spectrum = DirectionalSpectrum()
        
        if ai_model == 'xgboost':
            self.l3_ai = XGBoostSignalModel()
        else:
            self.l3_ai = MLPSignalModel()
        
        self.data_fetcher = DataFetcher()
        print(f"[SignalService] Initialized with {ai_model.upper()}")
    
    def generate_signal(self, symbol='BTC/USDT', timeframe='4h', train_if_needed=True):
        try:
            df = self.data_fetcher.fetch_ohlcv(symbol, timeframe, limit=500)
            if df is None or len(df) < 100:
                print("[SignalService] Insufficient data")
                return None
            
            if train_if_needed and not getattr(self.l3_ai, '_is_trained', False):
                print("[SignalService] Training AI model...")
                self.l3_ai.train(df)
            
            self.l1_bcd.fit(df)
            l1_vote = self.l1_bcd.get_directional_vote(df)
            l2_vote = self.l2_ema.get_directional_vote(df)
            l3_vote = self.l3_ai.get_directional_vote(df)
            l4_mult = self.l4_risk.get_multiplier(df)
            
            spectrum = self.spectrum.calculate(l1_vote, l2_vote, l3_vote, l4_mult)
            risk_params = self.l4_risk.get_risk_params(df)
            
            price = df['Close'].iloc[-1]
            
            return SignalResult(
                timestamp=datetime.utcnow(),
                symbol=symbol,
                price=price,
                l1_vote=l1_vote,
                l2_vote=l2_vote,
                l3_vote=l3_vote,
                l4_mult=l4_mult,
                directional_bias=spectrum.directional_bias,
                action=spectrum.action,
                conviction_pct=spectrum.conviction_pct,
                trade_gate=spectrum.trade_gate,
                position_size_pct=spectrum.position_size_pct,
                sl_pct=risk_params['sl_pct'],
                tp_pct=risk_params['tp_pct'],
                leverage=risk_params['leverage'],
                model_used=self.ai_model_type
            )
            
        except Exception as e:
            print(f"[SignalService] Error: {e}")
            import traceback
            traceback.print_exc()
            return None

## 10. Test: Generate Signal

In [ ]:
# Test dengan MLP
print("="*60)
print("TESTING SIGNAL GENERATION (MLP)")
print("="*60 + "\n")

service = SignalService(ai_model='mlp')
signal = service.generate_signal()

if signal:
    print(f"\n📊 Signal Generated:")
    print(f"  Symbol: {signal.symbol}")
    print(f"  Price: ${signal.price:,.2f}")
    print(f"\n🔢 Layer Votes:")
    print(f"  L1 (BCD):  {signal.l1_vote:+.3f} [30%]")
    print(f"  L2 (EMA):  {signal.l2_vote:+.3f} [25%]")
    print(f"  L3 (MLP):  {signal.l3_vote:+.3f} [45%] ← GATEKEEPER")
    print(f"  L4 (Risk): {signal.l4_mult:.3f} [multiplier]")
    print(f"\n📈 Spectrum:")
    print(f"  Directional Bias: {signal.directional_bias:+.3f}")
    print(f"  Action: {signal.action}")
    print(f"  Conviction: {signal.conviction_pct:.1f}%")
    print(f"  Gate: {signal.trade_gate}")
    print(f"\n⚙️ Risk Params:")
    print(f"  SL: {signal.sl_pct}% | TP: {signal.tp_pct}% | Lev: {signal.leverage}x")
    
    if signal.trade_gate == 'ACTIVE':
        print(f"\n✅ SIGNAL IS ACTIVE - Can Enter Trade")
    elif signal.trade_gate == 'ADVISORY':
        print(f"\n⚠️ ADVISORY - Reduce Size, Wait for Confirmation")
    else:
        print(f"\n❌ SUSPENDED - Do Not Trade")
else:
    print("❌ Failed to generate signal")

## 11. Experiment: Compare MLP vs XGBoost

In [ ]:
def compare_models_on_data(df):
    """Compare MLP and XGBoost on same dataset"""
    
    l1 = BayesianChangepointModel()
    l2 = EMASignalModel()
    spectrum = DirectionalSpectrum()
    
    models = {
        'mlp': MLPSignalModel(),
        'xgboost': XGBoostSignalModel()
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n{'='*60}")
        print(f"Testing {name.upper()}...")
        print(f"{'='*60}")
        
        # Train
        model.train(df)
        l1.fit(df)
        
        # Test on last 30%
        test_start = int(len(df) * 0.7)
        correct = 0
        total = 0
        biases = []
        
        for i in range(test_start, len(df) - 1):
            current_df = df.iloc[:i+1]
            
            l1_vote = l1.get_directional_vote(current_df)
            l2_vote = l2.get_directional_vote(current_df)
            l3_vote = model.get_directional_vote(current_df)
            
            raw_score = l1_vote * 0.30 + l2_vote * 0.25 + l3_vote * 0.45
            biases.append(raw_score)
            
            current_price = df['Close'].iloc[i]
            next_price = df['Close'].iloc[i+1]
            actual_up = next_price > current_price
            predicted_up = raw_score > 0
            
            if predicted_up == actual_up:
                correct += 1
            total += 1
        
        accuracy = correct / total * 100 if total > 0 else 0
        avg_bias = np.mean(np.abs(biases))
        
        results[name] = {
            'accuracy': accuracy,
            'avg_bias': avg_bias,
            'long_signals': sum(1 for b in biases if b > 0.2),
            'short_signals': sum(1 for b in biases if b < -0.2)
        }
        
        print(f"  Directional Accuracy: {accuracy:.1f}%")
        print(f"  Avg Bias Magnitude: {avg_bias:.4f}")
        print(f"  Long Signals: {results[name]['long_signals']}")
        print(f"  Short Signals: {results[name]['short_signals']}")
    
    return results

# Run comparison
print("\n" + "="*60)
print("MODEL COMPARISON: MLP vs XGBOOST")
print("="*60)

fetcher = DataFetcher()
df = fetcher.fetch_ohlcv('BTC/USDT', '4h', limit=500)

if df is not None:
    comparison_results = compare_models_on_data(df)
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    for name, res in comparison_results.items():
        print(f"\n{name.upper()}:")
        print(f"  Accuracy: {res['accuracy']:.1f}%")
        print(f"  Bias Mag: {res['avg_bias']:.4f}")
        print(f"  Signals: {res['long_signals']} Long / {res['short_signals']} Short")

## 12. Experiment: Walk-Forward Backtest

In [ ]:
def walk_forward_backtest(df, ai_model='mlp', verbose=True):
    """Run walk-forward backtest"""
    
    if ai_model == 'xgboost':
        l3 = XGBoostSignalModel()
    else:
        l3 = MLPSignalModel()
    
    l1 = BayesianChangepointModel()
    l2 = EMASignalModel()
    spectrum = DirectionalSpectrum()
    
    # Train on first 60%
    train_size = int(len(df) * 0.6)
    df_train = df.iloc[:train_size]
    
    print(f"[Backtest] Training on {len(df_train)} candles...")
    l3.train(df_train)
    
    # Test
    signals = []
    
    for i in range(train_size, len(df)):
        current_df = df.iloc[:i+1]
        
        if i % 12 == 0:
            l1.fit(current_df)
        
        l1_vote = l1.get_directional_vote(current_df)
        l2_vote = l2.get_directional_vote(current_df)
        l3_vote = l3.get_directional_vote(current_df)
        
        # Simplified: L4 = 1.0 for testing
        spectrum_result = spectrum.calculate(l1_vote, l2_vote, l3_vote, 1.0)
        
        signals.append({
            'timestamp': current_df.index[-1],
            'price': current_df['Close'].iloc[-1],
            'l1': l1_vote,
            'l2': l2_vote,
            'l3': l3_vote,
            'bias': spectrum_result.directional_bias,
            'action': spectrum_result.action,
            'gate': spectrum_result.trade_gate,
            'conviction': spectrum_result.conviction_pct
        })
        
        if verbose and i % 50 == 0:
            print(f"  [{i}/{len(df)}] bias={spectrum_result.directional_bias:+.3f}")
    
    return pd.DataFrame(signals)

# Run backtest
print("\n" + "="*60)
print("WALK-FORWARD BACKTEST")
print("="*60 + "\n")

fetcher = DataFetcher()
df = fetcher.fetch_ohlcv('BTC/USDT', '4h', limit=800)

if df is not None:
    results = walk_forward_backtest(df, ai_model='mlp', verbose=True)
    
    print("\n" + "="*60)
    print("BACKTEST ANALYSIS")
    print("="*60 + "\n")
    
    total = len(results)
    active = len(results[results['gate'] == 'ACTIVE'])
    advisory = len(results[results['gate'] == 'ADVISORY'])
    suspended = len(results[results['gate'] == 'SUSPENDED'])
    
    print(f"Total Signals: {total}")
    print(f"  🟢 ACTIVE: {active} ({active/total*100:.1f}%)")
    print(f"  🟡 ADVISORY: {advisory} ({advisory/total*100:.1f}%)")
    print(f"  🔴 SUSPENDED: {suspended} ({suspended/total*100:.1f}%)")
    
    print(f"\nBias Stats:")
    print(f"  Mean: {results['bias'].mean():+.4f}")
    print(f"  StdDev: {results['bias'].std():.4f}")
    print(f"  Max: {results['bias'].max():+.4f} / Min: {results['bias'].min():+.4f}")
    
    print(f"\nConviction: {results['conviction'].mean():.2f}% avg")